In [30]:
import sys
sys.path.insert(0, "../../run")
from run_config import REPO_PATH, SEASONS
sys.path.insert(1, f"{REPO_PATH}")

from typing import List
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Helper functions

In [61]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

def encode_and_onehot_transform_teams(season_dfs, home_col='home', away_col='away', date_col='date', n_first_matches=5):
    all_data = []
    team_match_counts = {}
    team_last_seen_season = set()

    for season_idx, df in enumerate(season_dfs):
        df = df.copy()
        df.sort_values(by=date_col, inplace=True)
        current_teams = set(df[home_col]).union(set(df[away_col]))

        is_new_home = []
        is_new_away = []
        encoded_home = []
        encoded_away = []
        both_new = []
        cold_start = []

        for i, row in df.iterrows():
            home_team = row[home_col]
            away_team = row[away_col]

            home_new = home_team not in team_last_seen_season
            away_new = away_team not in team_last_seen_season
            is_new_home.append(home_new)
            is_new_away.append(away_new)

            team_match_counts.setdefault(home_team, 0)
            team_match_counts.setdefault(away_team, 0)

            if home_new and team_match_counts[home_team] < n_first_matches:
                encoded_home.append('new_team_home')
            else:
                encoded_home.append(home_team)

            if away_new and team_match_counts[away_team] < n_first_matches:
                encoded_away.append('new_team_away')
            else:
                encoded_away.append(away_team)

            both_new_flag = home_new and away_new
            both_new.append(both_new_flag)

            cold_start_flag = both_new_flag and team_match_counts[home_team] == 0 and team_match_counts[away_team] == 0
            cold_start.append(cold_start_flag)

            # Update match counts
            team_match_counts[home_team] += 1
            team_match_counts[away_team] += 1

        df['encoded_home'] = encoded_home
        df['encoded_away'] = encoded_away
        df['is_new_home_team'] = is_new_home
        df['is_new_away_team'] = is_new_away
        df['both_teams_new'] = both_new
        df['is_cold_start_match'] = cold_start

        all_data.append(df)
        team_last_seen_season = current_teams

    final_df = pd.concat(all_data, ignore_index=True)

    # One-hot encode encoded team names
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    team_features = encoder.fit_transform(final_df[['encoded_home', 'encoded_away']])
    team_feature_names = encoder.get_feature_names_out(['encoded_home', 'encoded_away'])
    team_df = pd.DataFrame(team_features, columns=team_feature_names, index=final_df.index)

    return pd.concat([final_df[['is_new_home_team', 'is_new_away_team', 'both_teams_new', 'is_cold_start_match']], team_df], axis=1)


# Read Processed Data

In [38]:
processed_data_path='../../data/processed/premier_league/'

In [39]:
seasons=sorted(SEASONS)

In [40]:
data_dfs=[pd.read_csv(f"{processed_data_path}/{season}/all_data_df.csv") for season in seasons]

In [62]:
teams_encoding=encode_and_onehot_transform_teams(data_dfs)